# Electronics Few-Shot PCF on Colab (Qwen)

Runs Electronics-only few-shot PCF prediction using the codebase's canonical retrieval + prompt pipeline.

- Model: `Qwen/Qwen2.5-3B-Instruct` (as reported in `docs/main.tex`)
- Prompt source: `src/carbon/retrieval.py` (`OpenAILLMClient.system_prompt` + `build_few_shot_prompt`)
- Includes a first-pass test on ~50 products with structured logging before optional full run.



## 1) Mount Google Drive

Mounts Drive so logs and predictions persist outside the Colab runtime.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 2) Colab Setup and Repository Bootstrap

Clones the repo into Colab runtime, installs dependencies, and creates the Drive output folder.

In [ ]:
from pathlib import Path
import subprocess
import sys
import shutil

DRIVE_BASE = Path('/content/drive/MyDrive')
DRIVE_RUN_DIR = DRIVE_BASE / 'carbon-aware-recsys-colab' / 'pcf' / 'electronics_few_shot'
REPO_ROOT = Path('/content/carbon-aware-recsys')
REPO_URL = 'https://github.com/andersvestrum/carbon-aware-recsys.git'
REPO_BRANCH = 'main'

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(
    ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)],
    check=True,
)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'accelerate'], check=True)

DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
print({'repo_root': str(REPO_ROOT), 'drive_run_dir': str(DRIVE_RUN_DIR)})


## 3) Run Configuration and Logging

Sets model/category/run paths, output filenames, and logger handlers (console + Drive log file).

In [ ]:
import os
import sys
import time
import json
import logging
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(REPO_ROOT))

from src.carbon.retrieval import (
    OpenAILLMClient,
    TransformersLLMClient,
    PCFRetrievalEstimator,
    RetrievalConfig,
    build_few_shot_prompt,
    set_global_determinism,
)
from src.data.amazon_loader import load_meta

LLM_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
CATEGORY = 'electronics'
SEED = 42
TEST_LIMIT = 50
TOP_K = 5
NUM_THREADS = 4

OUTPUT_PREDICTIONS_TEST = DRIVE_RUN_DIR / 'predictions_electronics_few_shot_test50.csv'
OUTPUT_PREVIEW_TEST = DRIVE_RUN_DIR / 'predictions_first50_preview.csv'
OUTPUT_METADATA_TEST = DRIVE_RUN_DIR / 'run_metadata_test50.json'
OUTPUT_JSONL_TEST = DRIVE_RUN_DIR / 'predictions_log_test50.jsonl'

OUTPUT_PREDICTIONS_FULL = DRIVE_RUN_DIR / 'predictions_electronics_few_shot.csv'
OUTPUT_METADATA_FULL = DRIVE_RUN_DIR / 'run_metadata.json'
OUTPUT_JSONL_FULL = DRIVE_RUN_DIR / 'predictions_log.jsonl'

RUN_LOG = DRIVE_RUN_DIR / 'run.log'
LLM_CACHE = DRIVE_RUN_DIR / 'llm_prediction_cache.jsonl'

logger = logging.getLogger('electronics_few_shot_colab')
logger.setLevel(logging.INFO)
logger.handlers.clear()

fmt = logging.Formatter('%(asctime)s  %(levelname)s  %(message)s', datefmt='%H:%M:%S')
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(fmt)
file_handler = logging.FileHandler(RUN_LOG, mode='a', encoding='utf-8')
file_handler.setFormatter(fmt)

logger.addHandler(stream_handler)
logger.addHandler(file_handler)

logger.info('Configured run directory: %s', DRIVE_RUN_DIR)
logger.info('Model: %s | Category: %s', LLM_MODEL, CATEGORY)
print({'run_log': str(RUN_LOG), 'llm_cache': str(LLM_CACHE)})


## 4) Verify Canonical Prompt Source

Prints the exact system prompt and renders the few-shot template from `retrieval.py` for transparency.

In [ ]:
# Verify and display the exact prompt sources used from retrieval.py
print('System prompt from OpenAILLMClient.system_prompt:')
print('-' * 80)
print(OpenAILLMClient.system_prompt)
print('-' * 80)

sample_neighbors = [
    {'product_title': 'Stainless steel vacuum bottle 500ml', 'pcf': 4.1, 'similarity': 0.94, 'category': 'Kitchenware'},
    {'product_title': 'Insulated coffee tumbler 400ml', 'pcf': 2.8, 'similarity': 0.91, 'category': 'Kitchenware'},
    {'product_title': 'Aluminium sports bottle 600ml', 'pcf': 3.7, 'similarity': 0.88, 'category': 'Sporting goods'},
    {'product_title': 'Insulated food jar 300ml', 'pcf': 5.0, 'similarity': 0.85, 'category': 'Kitchenware'},
    {'product_title': 'Reusable travel mug 355ml', 'pcf': 1.4, 'similarity': 0.82, 'category': 'Kitchenware'},
]

print('\nRendered few-shot template from build_few_shot_prompt(...):')
print('-' * 80)
print(build_few_shot_prompt('Stainless steel double-wall travel mug 16 oz', sample_neighbors))
print('-' * 80)


## 5) Initialize Qwen Client and Retrieval Estimator

Loads the local Transformers LLM backend, sets deterministic config, and fits the Carbon Catalogue retrieval index.

In [ ]:
import torch
import transformers

transformers.logging.set_verbosity_error()

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

set_global_determinism(SEED, deterministic=True, num_threads=NUM_THREADS)

llm_client = TransformersLLMClient(
    model=LLM_MODEL,
    torch_dtype='float16',
    device_map='auto',
    max_new_tokens=384,
)

config = RetrievalConfig(
    top_k=TOP_K,
    random_seed=SEED,
    deterministic=True,
    num_threads=NUM_THREADS,
    llm_workers=1,
    device='cpu',
)

estimator = PCFRetrievalEstimator(config)
logger.info('Fitting Carbon Catalogue retrieval index...')
estimator.fit_carbon_catalogue()
logger.info('Carbon Catalogue fit complete.')


## 6) Helper Functions for Structured Logs

Defines helpers to shape preview tables and write row-level JSONL prediction logs.

In [ ]:
def _prediction_log_columns(df: pd.DataFrame, top_k: int) -> list[str]:
    cols = [
        'parent_asin',
        'title',
        'main_category',
        'store',
        'price',
        'source_category',
        'few_shot_llm_pcf',
        'neighbor_average_pcf',
        'pcf',
        'pcf_source',
    ]
    for rank in range(1, top_k + 1):
        cols.extend([
            f'neighbor_{rank}_id',
            f'neighbor_{rank}_title',
            f'neighbor_{rank}_category',
            f'neighbor_{rank}_pcf',
            f'neighbor_{rank}_similarity',
        ])
    return [c for c in cols if c in df.columns]


def _write_prediction_jsonl(df: pd.DataFrame, output_path: Path, top_k: int) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cols = _prediction_log_columns(df, top_k)
    records = df[cols].to_dict(orient='records')
    with output_path.open('w', encoding='utf-8') as handle:
        for rec in records:
            handle.write(json.dumps(rec, ensure_ascii=True) + '\n')


def _preview_frame(df: pd.DataFrame) -> pd.DataFrame:
    preview_cols = [
        'parent_asin',
        'title',
        'few_shot_llm_pcf',
        'pcf',
        'pcf_source',
        'neighbor_1_title',
        'neighbor_1_similarity',
        'neighbor_2_title',
        'neighbor_2_similarity',
    ]
    preview_cols = [c for c in preview_cols if c in df.columns]
    return df[preview_cols].copy()


## 7) Test Run on First 50 Electronics Products

Runs few-shot PCF prediction on a small sample and writes structured test artifacts for inspection.

In [ ]:
# Test run on first ~50 electronics products
logger.info('Loading electronics metadata...')
meta_electronics = load_meta(CATEGORY)
meta_test = meta_electronics.head(TEST_LIMIT).copy()
logger.info('Test rows selected: %d', len(meta_test))

t0 = time.time()
pred_test = estimator.predict_amazon_products(
    meta_test,
    llm_client=llm_client,
    llm_model_name=LLM_MODEL,
    llm_cache_path=LLM_CACHE,
    llm_limit=TEST_LIMIT,
    llm_cache_only=False,
    llm_methods=('few_shot_llm',),
)
elapsed_test_min = (time.time() - t0) / 60
logger.info('Test run complete in %.2f min', elapsed_test_min)

pred_test.to_csv(OUTPUT_PREDICTIONS_TEST, index=False)
preview_test = _preview_frame(pred_test)
preview_test.to_csv(OUTPUT_PREVIEW_TEST, index=False)
_write_prediction_jsonl(pred_test, OUTPUT_JSONL_TEST, TOP_K)

metadata_test = {
    'run_type': 'test50',
    'category': CATEGORY,
    'llm_model': LLM_MODEL,
    'llm_methods': ['few_shot_llm'],
    'rows': int(len(pred_test)),
    'seed': SEED,
    'top_k_neighbors': TOP_K,
    'elapsed_minutes': round(elapsed_test_min, 3),
    'output_predictions_csv': str(OUTPUT_PREDICTIONS_TEST),
    'output_preview_csv': str(OUTPUT_PREVIEW_TEST),
    'output_jsonl': str(OUTPUT_JSONL_TEST),
    'run_log': str(RUN_LOG),
    'llm_cache': str(LLM_CACHE),
}
with OUTPUT_METADATA_TEST.open('w', encoding='utf-8') as handle:
    json.dump(metadata_test, handle, indent=2)

print('Saved test artifacts:')
print(' -', OUTPUT_PREDICTIONS_TEST)
print(' -', OUTPUT_PREVIEW_TEST)
print(' -', OUTPUT_JSONL_TEST)
print(' -', OUTPUT_METADATA_TEST)

display(preview_test.head(20))


## 8) Optional Full Electronics Run

Toggle `RUN_FULL=True` to process the full Electronics metadata set after validating the test run.

In [ ]:
# Optional full electronics run
RUN_FULL = False

if RUN_FULL:
    logger.info('Starting full electronics few-shot run...')
    t0 = time.time()
    pred_full = estimator.predict_amazon_products(
        meta_electronics,
        llm_client=llm_client,
        llm_model_name=LLM_MODEL,
        llm_cache_path=LLM_CACHE,
        llm_limit=None,
        llm_cache_only=False,
        llm_methods=('few_shot_llm',),
    )
    elapsed_full_min = (time.time() - t0) / 60
    logger.info('Full run complete in %.2f min', elapsed_full_min)

    pred_full.to_csv(OUTPUT_PREDICTIONS_FULL, index=False)
    _write_prediction_jsonl(pred_full, OUTPUT_JSONL_FULL, TOP_K)

    metadata_full = {
        'run_type': 'full',
        'category': CATEGORY,
        'llm_model': LLM_MODEL,
        'llm_methods': ['few_shot_llm'],
        'rows': int(len(pred_full)),
        'seed': SEED,
        'top_k_neighbors': TOP_K,
        'elapsed_minutes': round(elapsed_full_min, 3),
        'output_predictions_csv': str(OUTPUT_PREDICTIONS_FULL),
        'output_jsonl': str(OUTPUT_JSONL_FULL),
        'run_log': str(RUN_LOG),
        'llm_cache': str(LLM_CACHE),
    }
    with OUTPUT_METADATA_FULL.open('w', encoding='utf-8') as handle:
        json.dump(metadata_full, handle, indent=2)

    print('Saved full-run artifacts:')
    print(' -', OUTPUT_PREDICTIONS_FULL)
    print(' -', OUTPUT_JSONL_FULL)
    print(' -', OUTPUT_METADATA_FULL)
else:
    print('RUN_FULL=False, skipped full run. Toggle RUN_FULL=True to execute full electronics pass.')


## 9) Output Locations and Live Logs

Shows exactly where outputs are written on mounted Drive and where to monitor live progress.

In [ ]:
# Where outputs are written while running
print('Drive output base:')
print(DRIVE_RUN_DIR)
print('\nLive progress appears in:')
print(' - notebook cell output (progress bars + logs)')
print(' -', RUN_LOG)

print('\nCurrent files in output dir:')
for p in sorted(DRIVE_RUN_DIR.glob('*')):
    print(' -', p.name)
